In [16]:
import pandas as pd
from pathlib import Path

print(Path.cwd())

c:\Users\rakes\Desktop\book recommender\notebooks


In [17]:
books = pd.read_csv(f"../data/processed/books_clean1.csv")


In [18]:
books.head(2)

,isbn13,isbn10,title,authors,categories,thumbnail,description,published_year,average_rating,num_pages,ratings_count,title_and_subtitle
0,9780002005883,0002005883,Gilead,Marilynne Robinson,Fiction,http://books.google.com/books/content?id=KQZCP...,A NOVEL THAT READERS and critics have been eag...,2004.0,3.85,247.0,361.0,Gilead
1,9780002261982,0002261987,Spider's Web,"Charles Osborne, Agatha Christie",Detective and mystery stories,http://books.google.com/books/content?id=gA5GP...,A new 'Christie for Christmas' -- a full-lengt...,2000.0,3.83,241.0,5164.0,Spider's Web: A Novel


In [19]:
books.columns

Index(['isbn13', 'isbn10', 'title', 'authors', 'categories', 'thumbnail',
       'description', 'published_year', 'average_rating', 'num_pages',
       'ratings_count', 'title_and_subtitle'],
      dtype='str')

In [20]:
books["description"].str.len().describe()

count    5197.000000
mean      485.290552
std       421.909000
min       124.000000
25%       220.000000
50%       291.000000
75%       646.000000
max      5786.000000
Name: description, dtype: float64

In [21]:

books["description"].str.split().str.len().describe()

count    5197.000000
mean       78.774485
std        68.532750
min        25.000000
25%        35.000000
50%        47.000000
75%       105.000000
max       920.000000
Name: description, dtype: float64

In [22]:
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [23]:
documents = []

for _, row in books.iterrows():
    documents.append(
        Document(
            page_content=row["description"],
            metadata={
                "isbn13": str(row["isbn13"]),
                "title": row["title_and_subtitle"],
                "authors": row["authors"],
                "categories": row["categories"],
                "thumbnail": row["thumbnail"],
                "average_rating": row["average_rating"],
                "published_year": row["published_year"],
            },
        )
    )

In [24]:
len(documents)

5197

In [25]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50,
    separators=["\n\n", "\n", ". ", " ", ""]
)

In [26]:
split_documents = text_splitter.split_documents(documents)

In [27]:
print(f"Original Documents: {len(documents)}")
print(f"Split Documents: {len(split_documents)}")

Original Documents: 5197
Split Documents: 8173


In [28]:
split_documents[0]

Document(metadata={'isbn13': '9780002005883', 'title': 'Gilead', 'authors': 'Marilynne Robinson', 'categories': 'Fiction', 'thumbnail': 'http://books.google.com/books/content?id=KQZCPgAACAAJ&printsec=frontcover&img=1&zoom=1&source=gbs_api', 'average_rating': 3.85, 'published_year': 2004.0}, page_content='A NOVEL THAT READERS and critics have been eagerly anticipating for over a decade, Gilead is an astonishingly imagined story of remarkable lives. John Ames is a preacher, the son of a preacher and the grandson (both maternal and paternal) of preachers. It’s 1956 in Gilead, Iowa, towards the end of the Reverend Ames’s life, and he is absorbed in recording his family’s story, a legacy for the young son he will never see grow up')

In [29]:
split_documents[1]

Document(metadata={'isbn13': '9780002005883', 'title': 'Gilead', 'authors': 'Marilynne Robinson', 'categories': 'Fiction', 'thumbnail': 'http://books.google.com/books/content?id=KQZCPgAACAAJ&printsec=frontcover&img=1&zoom=1&source=gbs_api', 'average_rating': 3.85, 'published_year': 2004.0}, page_content='. Haunted by his grandfather’s presence, John tells of the rift between his grandfather and his father: the elder, an angry visionary who fought for the abolitionist cause, and his son, an ardent pacifist. He is troubled, too, by his prodigal namesake, Jack (John Ames) Boughton, his best friend’s lost son who returns to Gilead searching for forgiveness and redemption')

In [30]:
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings

In [31]:
from langchain_huggingface import HuggingFaceEmbeddings

embedding_model = HuggingFaceEmbeddings(
    model_name="BAAI/bge-small-en-v1.5",
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True}
)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

c:\Users\rakes\Desktop\book recommender\env\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\rakes\.cache\huggingface\hub\models--BAAI--bge-small-en-v1.5. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/366 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [32]:
embedding_model

HuggingFaceEmbeddings(model_name='BAAI/bge-small-en-v1.5', cache_folder=None, model_kwargs={'device': 'cpu'}, encode_kwargs={'normalize_embeddings': True}, query_encode_kwargs={}, multi_process=False, show_progress=False)

In [33]:
from langchain_chroma import Chroma

vector_store = Chroma.from_documents(
    documents=split_documents,
    embedding=embedding_model,
    persist_directory="../chroma_db"
)

In [34]:
vector_store._collection.count()

8173

In [35]:
retriever = vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 5}
)

In [41]:
query = "A novel about fathers and sons, forgiveness and redemption"

results = retriever.invoke(query)

In [42]:
for i, doc in enumerate(results, 1):
    print(f"\n========== Result {i} ==========")
    print("Title:", doc.metadata["title"])
    print("Author:", doc.metadata["authors"])
    print("Category:", doc.metadata["categories"])
    print(doc.page_content[:300])


========== Result 1 ==========
Title: The Road
Author: Cormac McCarthy
Category: Fiction
In a novel set in an indefinite, futuristic, post-apocalyptic world, a father and his young son make their way through the ruins of a devastated American landscape, struggling to survive and preserve the last remnants of their own humanity. 250,000 first printing.

========== Result 2 ==========
Title: Caucasia
Author: Danzy Senna
Category: Fiction
A debut novel explores the complications of race through the story of two daughters--one light-skinned and the other dark-skinned--of a black father and a white mother, who become torn apart by racial allegiances. Reprint.

========== Result 3 ==========
Title: A Complicated Kindness
Author: Miriam Toews
Category: Fiction
A witty, beleaguered teenager whose family is shattered by fundamentalist Christianity balances grief and hope in this coming-of-age novel. Left alone with her sad, peculiar father, she spends her days piecing together why her mother a

In [43]:
def recommend_books(query, k=5):
    results = retriever.invoke(query)

    recommendations = []

    for doc in results:
        recommendations.append({
            "title": doc.metadata["title"],
            "author": doc.metadata["authors"],
            "category": doc.metadata["categories"],
            "thumbnail": doc.metadata["thumbnail"],
            "rating": doc.metadata["average_rating"],
            "description": doc.page_content
        })

    return recommendations

In [44]:
recommend_books("A novel about fathers and sons", k=5)

[{'title': 'The Road',
  'author': 'Cormac McCarthy',
  'category': 'Fiction',
  'thumbnail': 'http://books.google.com/books/content?id=eMq_lTq6fUgC&printsec=frontcover&img=1&zoom=1&source=gbs_api',
  'rating': 3.96,
  'description': 'In a novel set in an indefinite, futuristic, post-apocalyptic world, a father and his young son make their way through the ruins of a devastated American landscape, struggling to survive and preserve the last remnants of their own humanity. 250,000 first printing.'},
 {'title': 'White Teeth',
  'author': 'Zadie Smith',
  'category': 'Fiction',
  'thumbnail': 'http://books.google.com/books/content?id=EIjErXbzk3MC&printsec=frontcover&img=1&zoom=1&source=gbs_api',
  'rating': 3.76,
  'description': "In the author's words, this novel is an attempt at a comic family epic of little England into which an explosion of ethnic colour is injected. It tells the story of three families, one Indian, one white, one mixed, in North London and Oxford from World War II to 